# eEcoLiDAR Laserchicken grid to COPC and COG

This example calculates Laserchicken metrics on a regular 10 m by 10 m grid over the same 800 m by 800 m four-tile AHN area of interest. Target volumes are square `Cell(10)` columns with infinite height. Processing uses four memory-bounded 400 m by 400 m spatial partitions and full-resolution COPC environment points. Results are written as spatially partitioned COPC and Cloud-Optimized GeoTIFF collections in Amersfoort / RD New (EPSG:28992). Each feature has one COG tile per processing partition rather than one AoI-wide raster.

## Feature scope

Laserchicken 0.8.1 exposes 245 feature names. The seven eigenvector-derived outputs (`eigenv_*`, `normal_vector_*`, and `slope`) are excluded as requested. `echo_ratio` is also excluded because Laserchicken explicitly supports it only for circular infinite cylinders and raises an error for square cells. A further 116 metrics containing `normalized_height` are deliberately excluded: Laserchicken's built-in normalization grid is anchored to observed point minima rather than the CRS grid, and terrain-aware normalization should be evaluated separately. The remaining 121 raw-height, intensity, density, and classification metrics are calculated together.

In [1]:
from __future__ import annotations

import json
import os
from functools import partial
from pathlib import Path

import duckdb
import laserchicken
import numpy as np
import pandas as pd
import pdal
import pyarrow as pa
import pyarrow.parquet as pq
import rasterio
from distributed import Client, LocalCluster
from IPython.display import display
from laserchicken import compute_features, compute_neighborhoods
from laserchicken import kd_tree as lc_kd_tree
from laserchicken.feature_extractor.feature_extraction import list_feature_names
from laserchicken.volume_specification import Cell
from pyproj import CRS
from rasterio.transform import from_origin

from copc_dataset import DaskCopcExecutor, open_copc_dataset

REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / 'ahn5.parquet').exists():
    raise FileNotFoundError('Run this notebook from the UseCase_Ecology repository root')

CATALOG_PATH = REPOSITORY_ROOT / 'ahn5.parquet'
OUTPUT_ROOT = REPOSITORY_ROOT / 'outputs' / 'grid_10m_laserchicken'
COPC_DIRECTORY = OUTPUT_ROOT / 'copc'
COG_DIRECTORY = OUTPUT_ROOT / 'cog'
COPC_CATALOG_PATH = OUTPUT_ROOT / 'grid_10m_copc_catalog.parquet'
COG_MANIFEST_PATH = OUTPUT_ROOT / 'grid_10m_cog_collection.json'
COG_CATALOG_PATH = OUTPUT_ROOT / 'grid_10m_cog_catalog.parquet'
FEATURE_MAP_PATH = OUTPUT_ROOT / 'copc_feature_name_map.json'
for directory in (OUTPUT_ROOT, COPC_DIRECTORY, COG_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

In [2]:
dataset = open_copc_dataset(CATALOG_PATH)
RD_NEW = CRS.from_epsg(28992)
CELL_SIZE = 10.0
AOI_X = (125_600.0, 126_400.0)
AOI_Y = (485_600.0, 486_400.0)
GRID_COLUMNS = int((AOI_X[1] - AOI_X[0]) / CELL_SIZE)
GRID_ROWS = int((AOI_Y[1] - AOI_Y[0]) / CELL_SIZE)
PARTITION_COLUMNS = 2
PARTITION_ROWS = 2
TARGET_SHARD_SIZE = int(os.environ.get('LC_GRID_TARGET_SHARD_SIZE', '200'))
EXECUTION_BATCH_SIZE = int(os.environ.get('LC_GRID_EXECUTION_BATCH_SIZE', '2'))
SOURCE_QUERY_EPSILON = 0.001
READ_COLUMNS = ('X', 'Y', 'Z', 'Intensity', 'Classification')
PARTITION_FILTER = os.environ.get('LC_GRID_PARTITION_FILTER')

EIGEN_FEATURES = {
    'eigenv_1', 'eigenv_2', 'eigenv_3',
    'normal_vector_1', 'normal_vector_2', 'normal_vector_3', 'slope',
}
INCOMPATIBLE_CELL_FEATURES = {'echo_ratio'}
NORMALIZED_HEIGHT_FEATURES = {
    name for name in list_feature_names() if 'normalized_height' in name
}
FEATURE_NAMES = tuple(
    name for name in list_feature_names()
    if name not in EIGEN_FEATURES | INCOMPATIBLE_CELL_FEATURES | NORMALIZED_HEIGHT_FEATURES
)
COPC_FEATURE_NAMES = {name: name for name in FEATURE_NAMES}
assert len(NORMALIZED_HEIGHT_FEATURES) == 116
assert len(FEATURE_NAMES) == 121
assert all(len(name) <= 32 for name in COPC_FEATURE_NAMES.values())
assert len(set(COPC_FEATURE_NAMES.values())) == len(COPC_FEATURE_NAMES)
assert GRID_COLUMNS == GRID_ROWS == 80

display({
    'crs': RD_NEW.to_string(),
    'grid': (GRID_ROWS, GRID_COLUMNS),
    'target_cells': GRID_ROWS * GRID_COLUMNS,
    'laserchicken_metrics': len(FEATURE_NAMES),
    'excluded_eigen_metrics': sorted(EIGEN_FEATURES),
    'excluded_incompatible_metrics': sorted(INCOMPATIBLE_CELL_FEATURES),
    'excluded_normalized_height_metrics': len(NORMALIZED_HEIGHT_FEATURES),
})

{'crs': 'EPSG:28992',
 'grid': (80, 80),
 'target_cells': 6400,
 'laserchicken_metrics': 121,
 'excluded_eigen_metrics': ['eigenv_1',
  'eigenv_2',
  'eigenv_3',
  'normal_vector_1',
  'normal_vector_2',
  'normal_vector_3',
  'slope'],
 'excluded_incompatible_metrics': ['echo_ratio'],
 'excluded_normalized_height_metrics': 116}

## Build aligned target partitions

The national grid origin is represented here by the EPSG:28992 integer metre origin `(0, 0)`. This AoI's boundaries are exact multiples of 10 m, so each target is at a cell center. The environment selection spans every target center plus half a cell in each horizontal direction. Adjacent partitions may read shared boundary points, but target grid cells never overlap.

In [3]:
partition_width = GRID_COLUMNS // PARTITION_COLUMNS
partition_height = GRID_ROWS // PARTITION_ROWS
environment_selections = []
target_specifications = {}

for partition_row in range(PARTITION_ROWS):
    for partition_column in range(PARTITION_COLUMNS):
        partition_id = f'r{partition_row:02d}-c{partition_column:02d}'
        column_start = partition_column * partition_width
        column_stop = (partition_column + 1) * partition_width
        row_start = partition_row * partition_height
        row_stop = (partition_row + 1) * partition_height
        columns = np.arange(column_start, column_stop, dtype=np.uint32)
        rows = np.arange(row_start, row_stop, dtype=np.uint32)
        grid_columns, grid_rows = np.meshgrid(columns, rows)
        x = AOI_X[0] + (grid_columns.ravel() + 0.5) * CELL_SIZE
        y = AOI_Y[0] + (grid_rows.ravel() + 0.5) * CELL_SIZE
        selection = dataset.selection(
            partition_id,
            x=(x.min() - CELL_SIZE / 2, x.max() + CELL_SIZE / 2 - SOURCE_QUERY_EPSILON),
            y=(y.min() - CELL_SIZE / 2, y.max() + CELL_SIZE / 2 - SOURCE_QUERY_EPSILON),
            z=None,
            columns=READ_COLUMNS,
        )
        environment_selections.append(selection)
        target_specifications[partition_id] = {
            'x': x.astype(np.float64),
            'y': y.astype(np.float64),
            'grid_column': grid_columns.ravel(),
            'grid_row': grid_rows.ravel(),
            'output_path': str(COPC_DIRECTORY / f'{partition_id}.copc.laz'),
        }

environment_selections = tuple(environment_selections)
assert len(environment_selections) == 4
assert sum(spec['x'].size for spec in target_specifications.values()) == 6400
if PARTITION_FILTER:
    environment_selections = tuple(
        selection for selection in environment_selections if selection.id == PARTITION_FILTER
    )
    if not environment_selections:
        raise ValueError(f'Unknown partition filter: {PARTITION_FILTER}')
display(pd.DataFrame([
    {
        'partition': selection.id,
        'targets': target_specifications[selection.id]['x'].size,
        'min_x': selection.bounds.min_x, 'max_x': selection.bounds.max_x,
        'min_y': selection.bounds.min_y, 'max_y': selection.bounds.max_y,
        'candidate_copcs': len(dataset.candidate_sources(selection)),
    }
    for selection in environment_selections
]))

,partition,targets,min_x,max_x,min_y,max_y,candidate_copcs
0,r00-c00,1600,125600.0,125999.999,485600.0,485999.999,1
1,r00-c01,1600,126000.0,126399.999,485600.0,485999.999,1
2,r01-c00,1600,125600.0,125999.999,486000.0,486399.999,1
3,r01-c01,1600,126000.0,126399.999,486000.0,486399.999,1


## Worker-side feature calculation and output writing

Each worker materializes one full-resolution environment partition and reuses one environment KD-tree across 200-cell target shards. Environment queries use half-open grid ownership at AHN's 1 mm coordinate scale: lower/left edges are included and upper/right edges stop at `edge - 0.001`. This prevents zero-width reads from an adjacent kilometer COPC while assigning boundary points to exactly one partition. The core targets and their 121 feature arrays are written directly to one COPC and 121 partition COGs before the arrays leave the worker. Cells without source points remain `NaN` in COG output and are omitted from COPC output because LAS coordinates cannot represent a missing Z.

In [4]:
def environment_cloud_from_batches(batches):
    chunks = {name: [] for name in READ_COLUMNS}
    for batch in batches:
        for name in READ_COLUMNS:
            chunks[name].append(batch[name].to_numpy(zero_copy_only=False))
    arrays = {name: np.concatenate(values) for name, values in chunks.items()}
    return {
        'vertex': {
            'x': {'type': 'float64', 'data': arrays['X'].astype(np.float64, copy=False)},
            'y': {'type': 'float64', 'data': arrays['Y'].astype(np.float64, copy=False)},
            'z': {'type': 'float64', 'data': arrays['Z'].astype(np.float64, copy=False)},
            'intensity': {'type': 'uint16', 'data': arrays['Intensity']},
            'raw_classification': {'type': 'uint8', 'data': arrays['Classification']},
        }
    }


def target_cloud(x, y):
    return {
        'vertex': {
            'x': {'type': 'float64', 'data': x},
            'y': {'type': 'float64', 'data': y},
            'z': {'type': 'float64', 'data': np.zeros(x.size, dtype=np.float64)},
        }
    }


def retain_only_environment_tree():
    x_refs, y_refs, trees = lc_kd_tree.kd_tree_cache
    lc_kd_tree.kd_tree_cache = ([x_refs[0]], [y_refs[0]], [trees[0]])


def write_grid_copc(path, x, y, z, columns, rows, feature_values, feature_name_map):
    valid = np.isfinite(z)
    dtype = [
        ('X', '<f8'), ('Y', '<f8'), ('Z', '<f8'),
        ('GridColumn', '<u4'), ('GridRow', '<u4'),
    ] + [(feature_name_map[name], '<f4') for name in feature_values]
    array = np.empty(int(valid.sum()), dtype=dtype)
    array['X'], array['Y'], array['Z'] = x[valid], y[valid], z[valid]
    array['GridColumn'], array['GridRow'] = columns[valid], rows[valid]
    for name, values in feature_values.items():
        array[feature_name_map[name]] = values[valid]
    writer = {
        'type': 'writers.copc', 'filename': str(path),
        'a_srs': 'EPSG:28992', 'extra_dims': 'all',
        'scale_x': 0.001, 'scale_y': 0.001, 'scale_z': 0.001,
        'offset_x': 'auto', 'offset_y': 'auto', 'offset_z': 'auto',
        'software_id': 'eEcoLiDAR laserchicken', 'pdal_metadata': True, 'threads': 2,
    }
    written = pdal.Pipeline(json.dumps([writer]), arrays=[array]).execute()
    assert written == array.size
    return int(array.size)


def write_partition_cogs(directory, partition_id, x, y, feature_values, cell_size):
    width = len(np.unique(x))
    height = len(np.unique(y))
    transform = from_origin(
        x.min() - cell_size / 2, y.max() + cell_size / 2, cell_size, cell_size
    )
    bounds = [
        float(x.min() - cell_size / 2), float(y.min() - cell_size / 2),
        float(x.max() + cell_size / 2), float(y.max() + cell_size / 2),
    ]
    assets = []
    for name, values in feature_values.items():
        feature_directory = Path(directory) / name
        feature_directory.mkdir(parents=True, exist_ok=True)
        path = feature_directory / f'{partition_id}.tif'
        with rasterio.open(
            path, 'w', driver='COG', height=height, width=width, count=1,
            dtype='float32', crs='EPSG:28992', transform=transform, nodata=np.nan,
            compress='DEFLATE', blocksize=256, overview_resampling='average',
        ) as destination:
            destination.write(np.flipud(values.reshape(height, width)), 1)
            destination.set_band_description(1, name)
            destination.update_tags(
                feature=name, partition=partition_id,
                target_volume='10m square cell, infinite height',
                laserchicken_version=laserchicken.__version__, height_normalization='none',
            )
        assets.append({
            'id': f'{name}-{partition_id}', 'feature': name,
            'partition_id': partition_id, 'path': str(path), 'bbox': bounds,
            'width': width, 'height': height,
        })
    return assets


def process_grid_partition(
    batches, selection, dataset_metadata, *, target_specs, feature_names,
    feature_name_map, cell_size, target_shard_size, cog_directory
):
    del dataset_metadata
    specification = target_specs[selection.id]
    environment = environment_cloud_from_batches(batches)
    lc_kd_tree.initialize_cache()
    x, y = specification['x'], specification['y']
    output = {name: np.empty(x.size, dtype=np.float32) for name in feature_names}
    volume = Cell(cell_size)
    shard_count = 0
    maximum_neighborhood_memberships = 0
    try:
        lc_kd_tree.get_kdtree_for_pc(environment)
        for start in range(0, x.size, target_shard_size):
            stop = min(start + target_shard_size, x.size)
            targets = target_cloud(x[start:stop], y[start:stop])
            neighborhoods = list(compute_neighborhoods(environment, targets, volume))
            maximum_neighborhood_memberships = max(
                maximum_neighborhood_memberships, sum(len(indices) for indices in neighborhoods)
            )
            compute_features(
                environment, neighborhoods, targets, list(feature_names), volume, verbose=False
            )
            for name in feature_names:
                values = np.asarray(targets['vertex'][name]['data'], dtype=np.float32)
                output[name][start:stop] = np.where(np.isfinite(values), values, np.nan)
            shard_count += 1
            retain_only_environment_tree()
    finally:
        lc_kd_tree.initialize_cache()

    z = output['mean_z'].astype(np.float64)
    output_path = Path(specification['output_path'])
    point_count = write_grid_copc(
        output_path, x, y, z, specification['grid_column'], specification['grid_row'],
        output, feature_name_map,
    )
    cog_assets = write_partition_cogs(
        cog_directory, selection.id, x, y, output, cell_size
    )
    valid = np.isfinite(z)
    points = np.column_stack((x[valid], y[valid], z[valid]))
    minimum, maximum = points.min(axis=0), points.max(axis=0)
    environment_point_count = len(environment['vertex']['x']['data'])
    estimated_peak_bytes = (
        128 * environment_point_count
        + 48 * maximum_neighborhood_memberships
        + 16 * x.size * len(feature_names)
    )
    estimated_peak_gib = estimated_peak_bytes / 1024**3
    return {
        'partition_id': selection.id, 'output_path': str(output_path),
        'environment_points': environment_point_count,
        'target_count': x.size, 'copc_point_count': point_count,
        'target_shards': shard_count, 'copc_bbox': [*minimum.tolist(), *maximum.tolist()],
        'max_neighborhood_memberships': maximum_neighborhood_memberships,
        'estimated_peak_gib': estimated_peak_gib,
        'recommended_worker_gib': max(4, int(np.ceil(2 * estimated_peak_gib))),
        'cog_assets': cog_assets,
    }

## Execute in memory-bounded Dask waves

Two workers process two 400 m by 400 m partitions per wave. Recreating the cluster releases native memory retained by SciPy, Laserchicken, and PDAL allocators. The reported planning estimate assumes 128 bytes per environment point for Arrow, NumPy, and KD-tree residency; 48 bytes per neighborhood membership for Python/list storage; and 16 bytes per target-feature value for calculation and writer temporaries. `recommended_worker_gib` applies a 2x margin with a 4 GiB floor. It is a sizing estimate rather than an RSS measurement.

In [5]:
for old_path in COPC_DIRECTORY.glob('*.copc.laz'):
    old_path.unlink()
for old_path in COG_DIRECTORY.rglob('*.tif'):
    old_path.unlink()

connection = duckdb.connect(database=':memory:')
try:
    connection.execute('INSTALL pdal FROM community')
finally:
    connection.close()

processor = partial(
    process_grid_partition, target_specs=target_specifications,
    feature_names=FEATURE_NAMES, feature_name_map=COPC_FEATURE_NAMES,
    cell_size=CELL_SIZE, target_shard_size=TARGET_SHARD_SIZE,
    cog_directory=str(COG_DIRECTORY),
)
partition_results = {}
for batch_start in range(0, len(environment_selections), EXECUTION_BATCH_SIZE):
    selection_batch = environment_selections[batch_start:batch_start + EXECUTION_BATCH_SIZE]
    cluster = LocalCluster(
        n_workers=2, threads_per_worker=1, processes=True, dashboard_address=None
    )
    client = Client(cluster)
    try:
        executor = DaskCopcExecutor(
            client, reader_options={
                'requests': 2, 'batch_size': 65_536, 'install_extension': False,
            },
        )
        futures = executor.submit(dataset, selection_batch, processor=processor)
        partition_results.update({key: future.result() for key, future in futures.items()})
    finally:
        client.close()
        cluster.close()

ordered_results = [partition_results[selection.id] for selection in environment_selections]
display(pd.DataFrame([{
    key: result[key] for key in (
        'partition_id', 'environment_points', 'target_count',
        'copc_point_count', 'target_shards', 'estimated_peak_gib',
        'recommended_worker_gib',
    )
} for result in ordered_results]))

/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes:

/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/mwgrootes/miniforge3/envs/copc/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes: 17179869184
Start tree creation
Done with env tree creation
Done with target tree creation
Cylinder size in Bytes: 135716802.63507912
Memory size in Bytes:

,partition_id,environment_points,target_count,copc_point_count,target_shards,estimated_peak_gib,recommended_worker_gib
0,r00-c00,2434690,1600,1108,8,0.319148,4
1,r00-c01,937634,1600,484,8,0.131613,4
2,r01-c00,2296186,1600,1004,8,0.299664,4
3,r01-c01,1820756,1600,1484,8,0.235393,4


## Catalog and validate the partitioned COG collection

The workers have already written one COG per feature and spatial partition, so this client-side step handles metadata only. Grid rows were flipped to north-up raster storage during worker writes. Every tile has one `float32` band, `NaN` nodata, DEFLATE compression, and EPSG:28992 georeferencing. The Parquet tile catalog is the scalable collection index; the small JSON manifest records collection-level metadata. At national scale, downstream analysis can select catalog rows by feature and bounds, lazily open those tiles with Rasterio/Rioxarray, and combine them by coordinates with Xarray without constructing a nationwide in-memory target array.

An optional downstream pattern is:

```python
import rioxarray
import xarray as xr

tiles = pd.read_parquet(COG_CATALOG_PATH).query("feature == 'mean_z'")
arrays = [
    rioxarray.open_rasterio(OUTPUT_ROOT / href, chunks='auto').squeeze('band', drop=True)
    for href in tiles['href']
]
mean_z = xr.combine_by_coords(arrays, combine_attrs='drop_conflicts')
```

Rioxarray is not required to produce the collection and is therefore not imported by this notebook.

In [6]:
cog_records = []
for result in ordered_results:
    for asset in result['cog_assets']:
        path = Path(asset['path'])
        cog_records.append({
            **{key: value for key, value in asset.items() if key != 'path'},
            'href': str(path.relative_to(OUTPUT_ROOT)),
            'crs': 'EPSG:28992', 'cell_size': CELL_SIZE,
            'media_type': 'image/tiff; application=geotiff; profile=cloud-optimized',
        })
pq.write_table(pa.Table.from_pylist(cog_records), COG_CATALOG_PATH)

cog_manifest = {
    'id': 'eecolidar-laserchicken-grid-10m-cog', 'crs': 'EPSG:28992',
    'bbox': [AOI_X[0], AOI_Y[0], AOI_X[1], AOI_Y[1]],
    'cell_size': CELL_SIZE, 'partition_shape': [partition_height, partition_width],
    'target_volume': 'square cell with infinite height', 'height_normalization': 'none',
    'features': list(FEATURE_NAMES), 'partitions': len(ordered_results),
    'tile_count': len(cog_records), 'tile_catalog': COG_CATALOG_PATH.name,
}
COG_MANIFEST_PATH.write_text(json.dumps(cog_manifest, indent=2, sort_keys=True))
assert len(cog_records) == len(FEATURE_NAMES) * len(ordered_results)
assert len(list(COG_DIRECTORY.rglob('*.tif'))) == len(cog_records)
sample_cog_path = COG_DIRECTORY / 'mean_z' / f"{ordered_results[0]['partition_id']}.tif"
with rasterio.open(sample_cog_path) as mean_z_cog:
    assert mean_z_cog.crs == RD_NEW
    assert mean_z_cog.shape == (partition_height, partition_width)
    assert mean_z_cog.profile['driver'] == 'GTiff'
    assert mean_z_cog.tags(ns='IMAGE_STRUCTURE').get('LAYOUT') == 'COG'
display({
    'cog_manifest': str(COG_MANIFEST_PATH), 'cog_catalog': str(COG_CATALOG_PATH),
    'features': len(FEATURE_NAMES), 'partitions': len(ordered_results),
    'cog_tiles': len(cog_records),
    'bytes': sum(path.stat().st_size for path in COG_DIRECTORY.rglob('*.tif')),
})

{'cog_manifest': '/Users/mwgrootes/Projects/REPOS/HPC-DAT/UseCase_Ecology/outputs/grid_10m_laserchicken/grid_10m_cog_collection.json',
 'cog_catalog': '/Users/mwgrootes/Projects/REPOS/HPC-DAT/UseCase_Ecology/outputs/grid_10m_laserchicken/grid_10m_cog_catalog.parquet',
 'features': 121,
 'partitions': 4,
 'cog_tiles': 484,
 'bytes': 2616316}

## Catalog and validate the COPC collection

In [7]:
FEATURE_MAP_PATH.write_text(json.dumps(COPC_FEATURE_NAMES, indent=2, sort_keys=True))
copc_schema = [
    {'name': 'X', 'size': 8, 'type': 'floating'},
    {'name': 'Y', 'size': 8, 'type': 'floating'},
    {'name': 'Z', 'size': 8, 'type': 'floating'},
    {'name': 'GridColumn', 'size': 4, 'type': 'unsigned'},
    {'name': 'GridRow', 'size': 4, 'type': 'unsigned'},
] + [
    {'name': COPC_FEATURE_NAMES[name], 'size': 4, 'type': 'floating'}
    for name in FEATURE_NAMES
]
catalog_rows = []
for result in ordered_results:
    output_path = Path(result['output_path'])
    catalog_rows.append({
        'id': result['partition_id'],
        'assets': {'data': {'href': str(output_path.relative_to(OUTPUT_ROOT)), 'type': 'application/vnd.laszip+copc'}},
        'pc:count': result['copc_point_count'], 'pc:schemas': copc_schema,
        'proj:bbox': result['copc_bbox'],
        'proj:wkt2': RD_NEW.to_wkt(),
        'properties': {
            'processing:grid_cell_size': CELL_SIZE,
            'processing:target_volume': 'square cell with infinite height',
            'processing:height_normalization': 'none',
            'processing:feature_name_map': FEATURE_MAP_PATH.name,
            'processing:laserchicken_version': laserchicken.__version__,
        },
    })
pq.write_table(pa.Table.from_pylist(catalog_rows), COPC_CATALOG_PATH)
grid_copc_dataset = open_copc_dataset(COPC_CATALOG_PATH)
assert grid_copc_dataset.source_count == len(environment_selections)
assert grid_copc_dataset.point_count == sum(result['copc_point_count'] for result in ordered_results)
assert 'mean_z' in grid_copc_dataset.dimensions
assert grid_copc_dataset.crs.to_epsg() == 28992
display({
    'copc_catalog': str(COPC_CATALOG_PATH),
    'copc_files': grid_copc_dataset.source_count,
    'copc_points': grid_copc_dataset.point_count,
    'dimensions': len(grid_copc_dataset.dimensions),
    'bytes': sum(Path(result['output_path']).stat().st_size for result in ordered_results),
})

{'copc_catalog': '/Users/mwgrootes/Projects/REPOS/HPC-DAT/UseCase_Ecology/outputs/grid_10m_laserchicken/grid_10m_copc_catalog.parquet',
 'copc_files': 4,
 'copc_points': 4080,
 'dimensions': 126,
 'bytes': 1835555}

## Result

The output COPCs preserve explicit XYZ target locations and the 121 selected unnormalized-height Laserchicken metrics as numeric dimensions for subsequent spatial point-cloud processing. The COG collection contains 121 feature groups with one aligned raster tile per spatial partition; it does not construct or retain an AoI-wide feature raster in process memory. For larger areas, the Parquet catalog can be filtered before opening only the required feature tiles, and packages such as Rioxarray/Xarray can lazily combine those tiles by coordinates, backed by Dask where appropriate. Height normalization is intentionally deferred to a separate terrain-aware workflow. Partition boundaries and core ownership are processing details; neither output contains duplicate target cells.